# Dados IBGE

Livro municipal de território, população dos censos e PIB do SIDRA.
As definições das variáveis estão em **dictionary.csv**.
Os extratos brutos ficam em **data/** e não são versionados.


## Importações


## Base IBGE

In [1]:
import math
import os
import time
import requests
import pandas as pd
import numpy as np
from pathlib import Path
from typing import Optional, Dict, List, Union
from unidecode import unidecode


/Users/lucas.nascimento/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


## Configurações

In [2]:
def resolve_repo_root() -> Path:
    """Localiza a raiz do projeto que contém o diretório local de dados."""
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data").is_dir() and (candidate / "books").is_dir():
            return candidate
    return start


BASE_DIR = resolve_repo_root() / "data"
RAW_DIR = BASE_DIR / "raw" / "ibge"
TRUSTED_DIR = BASE_DIR / "trusted" / "ibge"
ANALYTICS_DIR = BASE_DIR / "analytics"

for path in [RAW_DIR, TRUSTED_DIR, ANALYTICS_DIR]:
    path.mkdir(parents=True, exist_ok=True)


## Funções

In [3]:
def get_json(url: str, params: Optional[dict] = None, timeout: int = 60) -> Union[list, dict]:
    """
    Faz a requisição GET e devolve o JSON interpretado (lista ou dicionário).
    """
    response = requests.get(url, params=params, timeout=timeout)
    response.raise_for_status()
    return response.json()


## Cadastro de municípios do IBGE

In [4]:
def baixar_municipios_ibge() -> pd.DataFrame:
    """
    Baixa todos os municípios brasileiros pela API de Localidades do IBGE.
    Retorna uma tabela padronizada com código IBGE, município, UF e região.
    """
    url = "https://servicodados.ibge.gov.br/api/v1/localidades/municipios"
    data = get_json(url)

    rows = []

    for item in data:
        # O IBGE pode devolver nulo em objetos aninhados; .get(chave, {}) ainda retorna None se a chave existir com valor nulo.
        microrregiao = item.get("microrregiao") or {}
        mesorregiao = microrregiao.get("mesorregiao") or {}
        uf = mesorregiao.get("UF") or {}
        regiao = uf.get("regiao") or {}

        regiao_imediata = item.get("regiao-imediata") or {}
        regiao_intermediaria = regiao_imediata.get("regiao-intermediaria") or {}

        rows.append({
            "codigo_ibge": str(item.get("id")).zfill(7),
            "municipio": item.get("nome"),
            "municipio_norm": unidecode(item.get("nome", "")).upper(),
            "uf": uf.get("sigla"),
            "codigo_uf": uf.get("id"),
            "nome_uf": uf.get("nome"),
            "regiao": regiao.get("nome"),
            "sigla_regiao": regiao.get("sigla"),
            "codigo_regiao": regiao.get("id"),
            "regiao_imediata": regiao_imediata.get("nome"),
            "codigo_regiao_imediata": regiao_imediata.get("id"),
            "regiao_intermediaria": regiao_intermediaria.get("nome"),
            "codigo_regiao_intermediaria": regiao_intermediaria.get("id"),
        })

    df = pd.DataFrame(rows)
    df = df.sort_values(["uf", "municipio"]).reset_index(drop=True)

    return df


In [5]:
dim_municipio = baixar_municipios_ibge()

dim_municipio.head()


,codigo_ibge,municipio,municipio_norm,uf,codigo_uf,nome_uf,regiao,sigla_regiao,codigo_regiao,regiao_imediata,codigo_regiao_imediata,regiao_intermediaria,codigo_regiao_intermediaria
0,1200013,Acrelândia,ACRELANDIA,AC,12.0,Acre,Norte,N,1.0,Rio Branco,120001,Rio Branco,1201
1,1200054,Assis Brasil,ASSIS BRASIL,AC,12.0,Acre,Norte,N,1.0,Brasiléia,120002,Rio Branco,1201
2,1200104,Brasiléia,BRASILEIA,AC,12.0,Acre,Norte,N,1.0,Brasiléia,120002,Rio Branco,1201
3,1200138,Bujari,BUJARI,AC,12.0,Acre,Norte,N,1.0,Rio Branco,120001,Rio Branco,1201
4,1200179,Capixaba,CAPIXABA,AC,12.0,Acre,Norte,N,1.0,Rio Branco,120001,Rio Branco,1201


In [6]:
dim_municipio.to_parquet(TRUSTED_DIR / "dim_municipio.parquet", index=False)
# dim_municipio.to_csv(TRUSTED_DIR / "dim_municipio.csv", index=False, encoding="utf-8")


In [7]:
print(dim_municipio.shape)
print(dim_municipio["codigo_ibge"].nunique())
print(dim_municipio.head())


(5571, 13)
5571
  codigo_ibge     municipio municipio_norm  uf  codigo_uf nome_uf regiao  \
0     1200013    Acrelândia     ACRELANDIA  AC       12.0    Acre  Norte   
1     1200054  Assis Brasil   ASSIS BRASIL  AC       12.0    Acre  Norte   
2     1200104     Brasiléia      BRASILEIA  AC       12.0    Acre  Norte   
3     1200138        Bujari         BUJARI  AC       12.0    Acre  Norte   
4     1200179      Capixaba       CAPIXABA  AC       12.0    Acre  Norte   

  sigla_regiao  codigo_regiao regiao_imediata  codigo_regiao_imediata  \
0            N            1.0      Rio Branco                  120001   
1            N            1.0       Brasiléia                  120002   
2            N            1.0       Brasiléia                  120002   
3            N            1.0      Rio Branco                  120001   
4            N            1.0      Rio Branco                  120001   

  regiao_intermediaria  codigo_regiao_intermediaria  
0           Rio Branco            

## Função consultar SIDRA

In [8]:
SIDRA_BASE_URL = "https://apisidra.ibge.gov.br/values"


def sidra_request(api_path: str, sleep_seconds: float = 0.5) -> pd.DataFrame:
    """
    Consulta a API SIDRA usando o caminho depois de /values.

    Exemplo de api_path:
        /t/5938/n6/all/v/37/p/last%201/d/v37%202
    """
    url = SIDRA_BASE_URL + api_path

    response = requests.get(url, timeout=120)
    response.raise_for_status()

    data = response.json()

    if not data or len(data) <= 1:
        return pd.DataFrame()

    header = data[0]
    rows = data[1:]

    df = pd.DataFrame(rows)

    # Renomeia colunas usando os nomes descritivos do SIDRA
    rename_map = {col: header.get(col, col) for col in df.columns}
    df = df.rename(columns=rename_map)

    time.sleep(sleep_seconds)

    return df


In [9]:
def find_column(
    df: pd.DataFrame,
    keywords: List[str],
    exclude_keywords: Optional[List[str]] = None,
) -> Optional[str]:
    """
    Devolve a primeira coluna cujo nome contém todas as palavras-chave (normalizadas em ASCII).
    Ignora colunas cujo nome contém alguma palavra de exclusão (por exemplo, evita "Sexo (Código)" ao buscar "sexo").
    """
    cols = list(df.columns)
    exclude_keywords = exclude_keywords or []

    for col in cols:
        col_norm = unidecode(str(col)).lower()
        if any(unidecode(ex).lower() in col_norm for ex in exclude_keywords):
            continue
        if all(unidecode(k).lower() in col_norm for k in keywords):
            return col

    return None


def parse_sidra_valor(series: pd.Series) -> pd.Series:
    """
    Converte a coluna Valor do SIDRA para número.
    Trata símbolos especiais como '...', '-', 'X' etc.
    """
    return (
        series.astype(str)
        .str.strip()
        .replace({
            "...": None,
            "-": None,
            "X": None,
            "x": None,
            "": None,
            "nan": None,
            "None": None,
        })
        .str.replace(",", ".", regex=False)
        .pipe(pd.to_numeric, errors="coerce")
    )


In [10]:
def padronizar_sidra_municipal(
    df: pd.DataFrame,
    nome_variavel: str
) -> pd.DataFrame:
    """
    Padroniza o retorno SIDRA em formato cidade-período-valor.
    """
    if df.empty:
        return df

    col_mun_cod = find_column(df, ["municipio", "codigo"])
    col_mun_nome = find_column(df, ["municipio"], exclude_keywords=["codigo"])
    col_ano = find_column(df, ["ano"]) or find_column(df, ["periodo"])
    col_valor = find_column(df, ["valor"])

    if col_mun_cod is None:
        raise ValueError(f"Coluna de código do município não encontrada. Colunas: {df.columns.tolist()}")

    if col_valor is None:
        raise ValueError(f"Coluna de valor não encontrada. Colunas: {df.columns.tolist()}")

    out = pd.DataFrame({
        "codigo_ibge": df[col_mun_cod].astype(str).str.zfill(7),
        "ano": df[col_ano].astype(str) if col_ano is not None else None,
        nome_variavel: parse_sidra_valor(df[col_valor])
    })

    if col_mun_nome is not None:
        out["municipio_sidra"] = df[col_mun_nome].astype(str)

    out = out.drop_duplicates()

    return out


## População estimada municipal — SIDRA tabela 6579

In [11]:
def baixar_populacao_estimada_sidra(periodo: str = "last%201") -> pd.DataFrame:
    """
    Baixa população residente estimada municipal pela tabela SIDRA 6579.

    periodo:
        "last%201" -> último período disponível
        "2021"     -> ano específico
        "all"      -> todos os períodos disponíveis
    """
    api_path = f"/t/6579/n6/all/v/9324/p/{periodo}/d/v9324%200"

    df_raw = sidra_request(api_path)
    df = padronizar_sidra_municipal(df_raw, "populacao_estimada")

    return df


In [12]:
pop_estimada = baixar_populacao_estimada_sidra(periodo="last%205")

pop_estimada.head()


,codigo_ibge,ano,populacao_estimada,municipio_sidra
0,1100015,2020,22728.0,Alta Floresta D'Oeste - RO
1,1100015,2021,22516.0,Alta Floresta D'Oeste - RO
2,1100015,2024,22853.0,Alta Floresta D'Oeste - RO
3,1100015,2025,22787.0,Alta Floresta D'Oeste - RO
4,1100015,2026,22724.0,Alta Floresta D'Oeste - RO


In [13]:
pop_estimada.to_parquet(TRUSTED_DIR / "sidra_populacao_estimada.parquet", index=False)


In [14]:
print(pop_estimada.shape)
print(pop_estimada["codigo_ibge"].nunique())
print(pop_estimada["populacao_estimada"].describe())


(27855, 4)
5571
count    2.785200e+04
mean     3.824824e+04
std      2.198867e+05
min      7.710000e+02
25%      5.391000e+03
50%      1.147750e+04
75%      2.547725e+04
max      1.239637e+07
Name: populacao_estimada, dtype: float64


## População do Censo 2022 - SIDRA tabela 608

In [15]:
def baixar_populacao_censo_2010_sidra() -> pd.DataFrame:
    """
    Baixa população residente municipal do Censo 2010.

    Tenta, em ordem:
        1. Tabela 608, variável 93, situação do domicílio Total e sexo Total;
        2. Tabela 202, variável 93, situação do domicílio Total e sexo Total;
        3. Variações com classificações invertidas.

    Retorna:
        codigo_ibge
        ano
        populacao_censo_2010
        municipio_sidra
    """

    tentativas = [
        # Tabela 608 - População residente, por situação do domicílio e sexo
        # c1 e c2 normalmente são situação do domicílio e sexo.
        "/t/608/n6/all/v/93/p/2010/c1/0/c2/0",
        "/t/608/n6/all/v/93/p/2010/c2/0/c1/0",

        # Tabela 202 - População residente, por sexo e situação do domicílio
        "/t/202/n6/all/v/93/p/2010/c1/0/c2/0",
        "/t/202/n6/all/v/93/p/2010/c2/0/c1/0",

        # Fallback: baixa todas as categorias e filtra depois
        "/t/608/n6/all/v/93/p/2010/c1/all/c2/all",
        "/t/202/n6/all/v/93/p/2010/c1/all/c2/all",
    ]

    erros = []

    df_raw = None
    api_path_usado = None

    for api_path in tentativas:
        try:
            print(f"Tentando: {api_path}")
            tmp = sidra_request(api_path)

            if tmp is not None and not tmp.empty:
                df_raw = tmp.copy()
                api_path_usado = api_path
                print(f"Sucesso com: {api_path}")
                break

        except Exception as e:
            erros.append((api_path, str(e)))
            print(f"Falhou: {api_path}")
            continue

    if df_raw is None or df_raw.empty:
        print("Todas as tentativas falharam.")
        for path, erro in erros:
            print("\nPATH:", path)
            print("ERRO:", erro[:500])
        return pd.DataFrame()

    # Se a tentativa trouxe todas as categorias, filtra Total.
    # Mantém compatibilidade com suas funções find_column e padronizar_sidra_municipal.
    col_sexo = find_column(
        df_raw,
        ["sexo"],
        exclude_keywords=["codigo"]
    )

    if col_sexo is not None:
        valores_sexo = (
            df_raw[col_sexo]
            .astype(str)
            .str.strip()
            .str.upper()
        )

        if "TOTAL" in set(valores_sexo):
            df_raw = df_raw[valores_sexo.eq("TOTAL")].copy()

    col_situacao = find_column(
        df_raw,
        ["situacao", "domicilio"],
        exclude_keywords=["codigo"]
    )

    if col_situacao is not None:
        valores_situacao = (
            df_raw[col_situacao]
            .astype(str)
            .str.strip()
            .str.upper()
        )

        if "TOTAL" in set(valores_situacao):
            df_raw = df_raw[valores_situacao.eq("TOTAL")].copy()

    df = padronizar_sidra_municipal(
        df_raw,
        "populacao_censo_2010"
    )

    df["ano"] = "2010"
    df["fonte_sidra_populacao_censo_2010"] = api_path_usado

    cols_ordem = [
        "codigo_ibge",
        "ano",
        "populacao_censo_2010",
        "municipio_sidra",
        "fonte_sidra_populacao_censo_2010",
    ]

    cols_ordem = [col for col in cols_ordem if col in df.columns]

    df = df[
        cols_ordem + [col for col in df.columns if col not in cols_ordem]
    ]

    df = (
        df
        .drop_duplicates(["codigo_ibge", "ano"])
        .sort_values("codigo_ibge")
        .reset_index(drop=True)
    )

    return df


In [16]:
populacao_censo_2010 = baixar_populacao_censo_2010_sidra()

populacao_censo_2010.head()


Tentando: /t/608/n6/all/v/93/p/2010/c1/0/c2/0
Sucesso com: /t/608/n6/all/v/93/p/2010/c1/0/c2/0


,codigo_ibge,ano,populacao_censo_2010,municipio_sidra,fonte_sidra_populacao_censo_2010
0,1100015,2010,24392,Alta Floresta D'Oeste - RO,/t/608/n6/all/v/93/p/2010/c1/0/c2/0
1,1100023,2010,90353,Ariquemes - RO,/t/608/n6/all/v/93/p/2010/c1/0/c2/0
2,1100031,2010,6313,Cabixi - RO,/t/608/n6/all/v/93/p/2010/c1/0/c2/0
3,1100049,2010,78574,Cacoal - RO,/t/608/n6/all/v/93/p/2010/c1/0/c2/0
4,1100056,2010,17029,Cerejeiras - RO,/t/608/n6/all/v/93/p/2010/c1/0/c2/0


In [17]:
populacao_censo_2010 = populacao_censo_2010.drop(columns=["fonte_sidra_populacao_censo_2010"])
populacao_censo_2010


,codigo_ibge,ano,populacao_censo_2010,municipio_sidra
0,1100015,2010,24392,Alta Floresta D'Oeste - RO
1,1100023,2010,90353,Ariquemes - RO
2,1100031,2010,6313,Cabixi - RO
3,1100049,2010,78574,Cacoal - RO
4,1100056,2010,17029,Cerejeiras - RO
...,...,...,...,...
5560,5222005,2010,12548,Vianópolis - GO
5561,5222054,2010,7371,Vicentinópolis - GO
5562,5222203,2010,4735,Vila Boa - GO
5563,5222302,2010,5145,Vila Propício - GO


In [18]:
print(populacao_censo_2010.shape)
if not populacao_censo_2010.empty and "codigo_ibge" in populacao_censo_2010.columns:
    print(populacao_censo_2010["codigo_ibge"].nunique())
if not populacao_censo_2010.empty and "populacao_censo_2010" in populacao_censo_2010.columns:
    print(populacao_censo_2010["populacao_censo_2010"].sum())


(5565, 4)
5565
190755799


In [19]:
populacao_censo_2010.to_parquet(TRUSTED_DIR / "sidra_populacao_censo_2010.parquet", index=False)


## População do Censo 2022 — SIDRA tabela 9514

In [20]:
def baixar_populacao_censo_2022_sidra() -> pd.DataFrame:
    """
    Baixa população residente do Censo 2022 pela tabela SIDRA 9514.

    Observação:
    A tabela 9514 possui classificações de sexo, idade e forma de declaração.
    Aqui usamos idade total e forma de declaração total.
    """
    api_path = (
        "/t/9514"
        "/n6/all"
        "/v/allxp"
        "/p/all"
        "/c2/all"
        "/c287/100362"
        "/c286/113635"
    )

    df_raw = sidra_request(api_path)

    if df_raw.empty:
        return df_raw

    # Use a coluna de rótulo "Sexo", e não "Sexo (Código)" (os códigos nunca são iguais a "TOTAL")
    col_sexo = find_column(df_raw, ["sexo"], exclude_keywords=["codigo"])
    if col_sexo is not None:
        df_raw = df_raw[df_raw[col_sexo].astype(str).str.strip().str.upper() == "TOTAL"].copy()

    df = padronizar_sidra_municipal(df_raw, "populacao_censo_2022")
    df["ano"] = "2022"

    return df


In [21]:
pop_censo_2022 = baixar_populacao_censo_2022_sidra()

pop_censo_2022.head()


,codigo_ibge,ano,populacao_censo_2022,municipio_sidra
0,1100015,2022,21494,Alta Floresta D'Oeste - RO
3,1100023,2022,96833,Ariquemes - RO
6,1100031,2022,5351,Cabixi - RO
9,1100049,2022,86887,Cacoal - RO
12,1100056,2022,15890,Cerejeiras - RO


In [22]:
pop_censo_2022.to_parquet(TRUSTED_DIR / "sidra_populacao_censo_2022.parquet", index=False)


In [23]:
print(pop_censo_2022.shape)
if not pop_censo_2022.empty and "codigo_ibge" in pop_censo_2022.columns:
    print(pop_censo_2022["codigo_ibge"].nunique())
if not pop_censo_2022.empty and "populacao_censo_2022" in pop_censo_2022.columns:
    print(pop_censo_2022["populacao_censo_2022"].sum())


(5570, 4)
5570
203080756


## Juntar Censos: 2010 e 2022

In [24]:
pop_censos = pop_censo_2022[
    [
        "codigo_ibge",
        "ano",
        "populacao_censo_2022",
    ]
].merge(
    populacao_censo_2010[
        [
            "codigo_ibge",
            "populacao_censo_2010",
        ]
    ],
    on="codigo_ibge",
    how="outer"
)

pop_censos.head()


,codigo_ibge,ano,populacao_censo_2022,populacao_censo_2010
0,1100015,2022,21494,24392.0
1,1100023,2022,96833,90353.0
2,1100031,2022,5351,6313.0
3,1100049,2022,86887,78574.0
4,1100056,2022,15890,17029.0


In [25]:
pop_censos["crescimento_populacao_censo_2010_2022"] = (
    pop_censos["populacao_censo_2022"]
    / pop_censos["populacao_censo_2010"].replace(0, np.nan)
    - 1
)

pop_censos["cagr_populacao_censo_2010_2022"] = np.where(
    (pop_censos["populacao_censo_2010"] > 0)
    & (pop_censos["populacao_censo_2022"] > 0),
    (
        pop_censos["populacao_censo_2022"]
        / pop_censos["populacao_censo_2010"]
    ) ** (1 / 12) - 1,
    np.nan
)


In [26]:
pop_censos[
    [
        "codigo_ibge",
        "populacao_censo_2010",
        "populacao_censo_2022",
        "crescimento_populacao_censo_2010_2022",
        "cagr_populacao_censo_2010_2022",
    ]
].head()


,codigo_ibge,populacao_censo_2010,populacao_censo_2022,crescimento_populacao_censo_2010_2022,cagr_populacao_censo_2010_2022
0,1100015,24392.0,21494,-0.118809,-0.010485
1,1100023,90353.0,96833,0.071719,0.005789
2,1100031,6313.0,5351,-0.152384,-0.013683
3,1100049,78574.0,86887,0.105798,0.008416
4,1100056,17029.0,15890,-0.066886,-0.005752


<!-- ## PIB Municipal — SIDRA tabela 5938 -->

In [27]:
pop_censos.to_parquet(
    TRUSTED_DIR / "populacao_censos_2010_2022_municipio.parquet",
    index=False
)


## PIB Municipal

In [28]:
def baixar_pib_municipal_total(periodo: str = "last%201") -> pd.DataFrame:
    """
    Baixa PIB municipal total a preços correntes pela SIDRA 5938.

    Variável 37:
        Produto Interno Bruto a preços correntes.
    """
    api_path = f"/t/5938/n6/all/v/37/p/{periodo}/d/v37%202"

    df_raw = sidra_request(api_path)
    df = padronizar_sidra_municipal(df_raw, "pib_corrente")

    return df


In [29]:
pib_total = baixar_pib_municipal_total(periodo="last%205")

pib_total.head()


,codigo_ibge,ano,pib_corrente,municipio_sidra
0,1100015,2019,495775.15,Alta Floresta D'Oeste - RO
1,1100015,2020,570241.61,Alta Floresta D'Oeste - RO
2,1100015,2021,734467.02,Alta Floresta D'Oeste - RO
3,1100015,2022,919520.27,Alta Floresta D'Oeste - RO
4,1100015,2023,1046342.55,Alta Floresta D'Oeste - RO


In [30]:
pib_total.to_parquet(TRUSTED_DIR / "sidra_pib_total.parquet", index=False)


## Médias Móveis - PIB

In [31]:
def criar_variaveis_ano_a_ano(
    df: pd.DataFrame,
    col_id: str = "codigo_ibge",
    col_ano: str = "ano",
    col_valor: str = "pib_corrente",
    prefixo: str = "pib_corrente"
) -> pd.DataFrame:
    """
    Cria variáveis ano contra ano para uma base em painel município x ano.

    Espera uma base com:
        codigo_ibge | ano | variável numérica

    Exemplo:
        codigo_ibge | ano | pib_corrente
    """

    out = df.copy()

    # Padronização básica
    out[col_id] = (
        out[col_id]
        .astype("string")
        .str.replace(r"\D", "", regex=True)
        .str.zfill(7)
    )

    out[col_ano] = pd.to_numeric(out[col_ano], errors="coerce").astype("Int64")
    out[col_valor] = pd.to_numeric(out[col_valor], errors="coerce")

    # Ordenação correta para cálculo temporal
    out = out.sort_values([col_id, col_ano]).reset_index(drop=True)

    # Validação de duplicidade
    duplicadas = out.duplicated([col_id, col_ano]).sum()

    if duplicadas > 0:
        print(f"Atenção: foram encontradas {duplicadas} duplicidades por {col_id}-{col_ano}.")
        print("Consolidando por média da variável numérica.")

        cols_first = [
            col for col in out.columns
            if col not in [col_valor]
        ]

        agg_dict = {col: "first" for col in cols_first if col not in [col_id, col_ano]}
        agg_dict[col_valor] = "mean"

        out = (
            out
            .groupby([col_id, col_ano], as_index=False)
            .agg(agg_dict)
            .sort_values([col_id, col_ano])
            .reset_index(drop=True)
        )

    g = out.groupby(col_id, group_keys=False)

    # Lags
    out[f"{prefixo}_lag_1y"] = g[col_valor].shift(1)
    out[f"{prefixo}_lag_2y"] = g[col_valor].shift(2)
    out[f"{prefixo}_lag_3y"] = g[col_valor].shift(3)
    # out[f"{prefixo}_lag_4y"] = g[col_valor].shift(4)

    # Variação absoluta
    out[f"{prefixo}_var_abs_1y"] = (
        out[col_valor] - out[f"{prefixo}_lag_1y"]
    )

    out[f"{prefixo}_var_abs_2y"] = (
        out[col_valor] - out[f"{prefixo}_lag_2y"]
    )

    out[f"{prefixo}_var_abs_3y"] = (
        out[col_valor] - out[f"{prefixo}_lag_3y"]
    )

    # Crescimento percentual ano contra ano
    out[f"{prefixo}_crescimento_1y"] = (
        out[col_valor] / out[f"{prefixo}_lag_1y"] - 1
    )

    out[f"{prefixo}_crescimento_2y_acumulado"] = (
        out[col_valor] / out[f"{prefixo}_lag_2y"] - 1
    )

    out[f"{prefixo}_crescimento_3y_acumulado"] = (
        out[col_valor] / out[f"{prefixo}_lag_3y"] - 1
    )

    # CAGR: crescimento médio anual composto
    out[f"{prefixo}_cagr_2y"] = np.where(
        (out[col_valor] > 0) & (out[f"{prefixo}_lag_2y"] > 0),
        (out[col_valor] / out[f"{prefixo}_lag_2y"]) ** (1 / 2) - 1,
        np.nan
    )

    out[f"{prefixo}_cagr_3y"] = np.where(
        (out[col_valor] > 0) & (out[f"{prefixo}_lag_3y"] > 0),
        (out[col_valor] / out[f"{prefixo}_lag_3y"]) ** (1 / 3) - 1,
        np.nan
    )

    # Log e variação logarítmica
    out[f"log_{prefixo}"] = np.where(
        out[col_valor] > 0,
        np.log(out[col_valor]),
        np.nan
    )

    out[f"{prefixo}_delta_log_1y"] = (
        out[f"log_{prefixo}"] - g[f"log_{prefixo}"].shift(1)
    )

    # Aceleração: crescimento atual menos crescimento anterior
    out[f"{prefixo}_crescimento_1y_lag_1y"] = (
        g[f"{prefixo}_crescimento_1y"].shift(1)
    )

    out[f"{prefixo}_aceleracao_crescimento_1y"] = (
        out[f"{prefixo}_crescimento_1y"]
        - out[f"{prefixo}_crescimento_1y_lag_1y"]
    )

    # Médias móveis
    out[f"{prefixo}_media_movel_3y"] = (
        g[col_valor]
        .rolling(window=3, min_periods=2)
        .mean()
        .reset_index(level=0, drop=True)
    )

    out[f"{prefixo}_desvio_movel_3y"] = (
        g[col_valor]
        .rolling(window=3, min_periods=2)
        .std()
        .reset_index(level=0, drop=True)
    )

    out[f"{prefixo}_coef_var_movel_3y"] = (
        out[f"{prefixo}_desvio_movel_3y"]
        / out[f"{prefixo}_media_movel_3y"].replace(0, np.nan)
    )

    # PIB relativo ao primeiro ano disponível do município
    primeiro_valor = g[col_valor].transform("first")

    out[f"{prefixo}_indice_base_primeiro_ano"] = (
        out[col_valor] / primeiro_valor * 100
    )

    out[f"{prefixo}_crescimento_acumulado_primeiro_ano"] = (
        out[col_valor] / primeiro_valor - 1
    )

    # Ranking e percentil dentro de cada ano
    out[f"{prefixo}_rank_ano"] = (
        out.groupby(col_ano)[col_valor]
        .rank(method="min", ascending=False)
    )

    out[f"{prefixo}_percentil_ano"] = (
        out.groupby(col_ano)[col_valor]
        .rank(pct=True, ascending=True)
    )

    # Participação no PIB total dos municípios daquele ano
    total_ano = out.groupby(col_ano)[col_valor].transform("sum")

    out[f"{prefixo}_share_total_ano"] = (
        out[col_valor] / total_ano.replace(0, np.nan)
    )

    return out


In [32]:
pib_total_var = criar_variaveis_ano_a_ano(
    df=pib_total,
    col_id="codigo_ibge",
    col_ano="ano",
    col_valor="pib_corrente",
    prefixo="pib_corrente"
)

pib_total_var.head(10)


,codigo_ibge,ano,pib_corrente,municipio_sidra,pib_corrente_lag_1y,pib_corrente_lag_2y,pib_corrente_lag_3y,pib_corrente_var_abs_1y,pib_corrente_var_abs_2y,pib_corrente_var_abs_3y,...,pib_corrente_crescimento_1y_lag_1y,pib_corrente_aceleracao_crescimento_1y,pib_corrente_media_movel_3y,pib_corrente_desvio_movel_3y,pib_corrente_coef_var_movel_3y,pib_corrente_indice_base_primeiro_ano,pib_corrente_crescimento_acumulado_primeiro_ano,pib_corrente_rank_ano,pib_corrente_percentil_ano,pib_corrente_share_total_ano
0,1100015,2019,495775.15,Alta Floresta D'Oeste - RO,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,100.000000,0.000000,1438.0,0.742011,0.000067
1,1100015,2020,570241.61,Alta Floresta D'Oeste - RO,495775.15,NaN,NaN,74466.46,NaN,NaN,...,NaN,NaN,5.330084e+05,52655.738837,0.098790,115.020208,0.150202,1419.0,0.745422,0.000075
2,1100015,2021,734467.02,Alta Floresta D'Oeste - RO,570241.61,495775.15,NaN,164225.41,238691.87,NaN,...,0.150202,0.137791,6.001613e+05,122126.333332,0.203489,148.145186,0.481452,1322.0,0.762837,0.000081
3,1100015,2022,919520.27,Alta Floresta D'Oeste - RO,734467.02,570241.61,495775.15,185053.25,349278.66,423745.12,...,0.287993,-0.036037,7.414096e+05,174742.798114,0.235690,185.471230,0.854712,1200.0,0.784740,0.000091
4,1100015,2023,1046342.55,Alta Floresta D'Oeste - RO,919520.27,734467.02,570241.61,126822.28,311875.53,476100.94,...,0.251956,-0.114034,9.001099e+05,156841.184139,0.174247,211.051835,1.110518,1184.0,0.787612,0.000096
5,1100023,2019,2579277.74,Ariquemes - RO,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,100.000000,0.000000,392.0,0.929803,0.000349
6,1100023,2020,2817330.87,Ariquemes - RO,2579277.74,NaN,NaN,238053.13,NaN,NaN,...,NaN,NaN,2.698304e+06,168328.982506,0.062383,109.229449,0.092294,380.0,0.931957,0.000370
7,1100023,2021,3211294.28,Ariquemes - RO,2817330.87,2579277.74,NaN,393963.41,632016.54,NaN,...,0.092294,0.047541,2.869301e+06,319197.266058,0.111246,124.503625,0.245036,404.0,0.927648,0.000356
8,1100023,2022,3809355.08,Ariquemes - RO,3211294.28,2817330.87,2579277.74,598060.80,992024.21,1230077.34,...,0.139836,0.046401,3.279327e+06,499499.069425,0.152318,147.690767,0.476908,388.0,0.930521,0.000378
9,1100023,2023,4383815.34,Ariquemes - RO,3809355.08,3211294.28,2817330.87,574460.26,1172521.06,1566484.47,...,0.186237,-0.035434,3.801488e+06,586300.114697,0.154229,169.962904,0.699629,359.0,0.935727,0.000401


In [33]:
pib_total_var[pib_total_var["ano"] == 2023]


,codigo_ibge,ano,pib_corrente,municipio_sidra,pib_corrente_lag_1y,pib_corrente_lag_2y,pib_corrente_lag_3y,pib_corrente_var_abs_1y,pib_corrente_var_abs_2y,pib_corrente_var_abs_3y,...,pib_corrente_crescimento_1y_lag_1y,pib_corrente_aceleracao_crescimento_1y,pib_corrente_media_movel_3y,pib_corrente_desvio_movel_3y,pib_corrente_coef_var_movel_3y,pib_corrente_indice_base_primeiro_ano,pib_corrente_crescimento_acumulado_primeiro_ano,pib_corrente_rank_ano,pib_corrente_percentil_ano,pib_corrente_share_total_ano
4,1100015,2023,1.046343e+06,Alta Floresta D'Oeste - RO,9.195203e+05,7.344670e+05,5.702416e+05,126822.28,311875.53,476100.94,...,0.251956,-0.114034,9.001099e+05,1.568412e+05,0.174247,211.051835,1.110518,1184.0,0.787612,0.000096
9,1100023,2023,4.383815e+06,Ariquemes - RO,3.809355e+06,3.211294e+06,2.817331e+06,574460.26,1172521.06,1566484.47,...,0.186237,-0.035434,3.801488e+06,5.863001e+05,0.154229,169.962904,0.699629,359.0,0.935727,0.000401
14,1100031,2023,3.008321e+05,Cabixi - RO,2.897827e+05,2.384141e+05,1.671528e+05,11049.43,62418.01,133679.27,...,0.215459,-0.177329,2.763430e+05,3.330873e+04,0.120534,214.916676,1.149167,2848.0,0.488869,0.000027
19,1100049,2023,3.848468e+06,Cacoal - RO,3.195489e+06,2.792506e+06,2.518845e+06,652978.41,1055961.99,1329622.90,...,0.144309,0.060035,3.278821e+06,5.328903e+05,0.162525,170.140894,0.701409,397.0,0.928905,0.000352
24,1100056,2023,1.006389e+06,Cerejeiras - RO,9.030985e+05,7.430617e+05,6.006303e+05,103290.90,263327.76,405759.09,...,0.215375,-0.101001,8.841832e+05,1.326790e+05,0.150058,198.697126,0.986971,1214.0,0.782226,0.000092
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27829,5222005,2023,9.149434e+05,Vianópolis - GO,9.606718e+05,8.580161e+05,5.590581e+05,-45728.36,56927.36,355885.37,...,0.119643,-0.167244,9.112104e+05,5.142957e+04,0.056441,187.003681,0.870037,1301.0,0.766607,0.000084
27834,5222054,2023,6.775996e+05,Vicentinópolis - GO,7.204436e+05,4.802101e+05,4.197570e+05,-42844.03,197389.51,257842.63,...,0.500268,-0.559737,6.260844e+05,1.281343e+05,0.204660,207.522512,1.075225,1623.0,0.708797,0.000062
27839,5222203,2023,2.018464e+05,Vila Boa - GO,1.948026e+05,1.799239e+05,1.261498e+05,7043.84,21922.58,75696.65,...,0.082695,-0.046536,1.921910e+05,1.119220e+04,0.058235,179.884968,0.798850,3509.0,0.370197,0.000018
27844,5222302,2023,4.442092e+05,Vila Propício - GO,4.575121e+05,3.080351e+05,2.179354e+05,-13302.96,136174.08,226273.75,...,0.485260,-0.514337,4.032521e+05,8.272820e+04,0.205153,213.567856,1.135679,2167.0,0.611131,0.000041


## Salvar

In [34]:
pib_total_var[pib_total_var["ano"] == 2023].to_parquet(TRUSTED_DIR / "sidra_pib_anoxano.parquet", index=False)


## PIB Setorial

In [35]:
PIB_SETORIAL_VARS = {
    "vab_agropecuaria": 513,
    "vab_industria": 517,
    "vab_servicos": 525,
    "vab_adm_publica": 543,
    "impostos_liquidos": 498,
}


In [36]:
def baixar_pib_setorial(periodo: str = "2021") -> pd.DataFrame:
    """
    Baixa variáveis setoriais do PIB municipal pela tabela SIDRA 5938.

    Use período específico, por exemplo 2021, pois a edição 2022-2023
    pode não ter abertura setorial tradicional.
    """
    dfs = []

    for nome_variavel, codigo_variavel in PIB_SETORIAL_VARS.items():
        api_path = (
            f"/t/5938"
            f"/n6/all"
            f"/v/{codigo_variavel}"
            f"/p/{periodo}"
            f"/d/v{codigo_variavel}%202"
        )

        df_raw = sidra_request(api_path)
        df = padronizar_sidra_municipal(df_raw, nome_variavel)
        if df.empty or not {"codigo_ibge", "ano"}.issubset(df.columns):
            raise ValueError(
                f"Resposta vazia ou inválida do SIDRA para a variável {codigo_variavel} ({nome_variavel})."
            )

        dfs.append(df[["codigo_ibge", "ano", nome_variavel]])

    base = dfs[0]

    for df in dfs[1:]:
        base = base.merge(df, on=["codigo_ibge", "ano"], how="outer")

    return base


In [37]:
pib_setorial = baixar_pib_setorial(periodo="last%205")

pib_setorial.head()


,codigo_ibge,ano,vab_agropecuaria,vab_industria,vab_servicos,vab_adm_publica,impostos_liquidos
0,1100015,2019,162668.42,14468.34,158972.74,29351.20,466423.95
1,1100015,2020,203394.42,20714.61,160790.58,35108.98,535132.63
2,1100015,2021,311468.68,27845.50,173121.63,49596.51,684870.51
3,1100015,2022,NaN,NaN,NaN,NaN,NaN
4,1100015,2023,NaN,NaN,NaN,NaN,NaN


In [38]:
pib_setorial_2021 = pib_setorial[pib_setorial["ano"] <= "2021"]
pib_setorial_2021


,codigo_ibge,ano,vab_agropecuaria,vab_industria,vab_servicos,vab_adm_publica,impostos_liquidos
0,1100015,2019,162668.42,14468.34,1.589727e+05,29351.20,4.664240e+05
1,1100015,2020,203394.42,20714.61,1.607906e+05,35108.98,5.351326e+05
2,1100015,2021,311468.68,27845.50,1.731216e+05,49596.51,6.848705e+05
5,1100023,2019,170830.70,284066.16,7.037565e+05,284379.78,2.294898e+06
6,1100023,2020,199722.71,404715.25,7.102664e+05,295644.71,2.521686e+06
...,...,...,...,...,...,...,...
27841,5222302,2020,115113.53,26920.91,3.285523e+04,8560.35,2.093750e+05
27842,5222302,2021,157949.39,57342.31,3.314975e+04,12865.04,2.951700e+05
27845,5300108,2019,992393.58,9453608.03,1.072192e+08,30686607.65,2.429271e+08
27846,5300108,2020,1623976.91,10942472.57,1.112670e+08,25466227.78,2.403811e+08


In [39]:
pib_setorial_2021.to_parquet(TRUSTED_DIR / "sidra_pib_setorial.parquet", index=False)


## Ano vs Ano: PIB Setorial

In [40]:
def criar_variaveis_ano_a_ano_multiplas(
    df: pd.DataFrame,
    col_id: str = "codigo_ibge",
    col_ano: str = "ano",
    cols_valores: Optional[List[str]] = None,
    criar_logs: bool = True,
    criar_ranking_ano: bool = True,
    criar_rolling: bool = True
) -> pd.DataFrame:
    """
    Cria variáveis ano contra ano para múltiplas colunas numéricas em painel município x ano.

    Espera uma base no formato:
        codigo_ibge | ano | vab_agropecuaria | vab_industria | ...

    Retorna a base original acrescida de:
        - lags
        - variações absolutas
        - crescimentos percentuais
        - CAGR
        - delta log
        - aceleração do crescimento
        - médias móveis
        - rankings anuais
        - participação no total anual
    """

    out = df.copy()

    # Padronização das chaves
    out[col_id] = (
        out[col_id]
        .astype("string")
        .str.replace(r"\D", "", regex=True)
        .str.zfill(7)
    )

    out[col_ano] = pd.to_numeric(out[col_ano], errors="coerce").astype("Int64")

    # Identifica automaticamente colunas numéricas, se não forem passadas
    if cols_valores is None:
        cols_valores = []

        for col in out.columns:
            if col in [col_id, col_ano]:
                continue

            serie_num = pd.to_numeric(out[col], errors="coerce")

            if serie_num.notna().mean() > 0.8:
                cols_valores.append(col)

    # Garante conversão numérica
    for col in cols_valores:
        out[col] = pd.to_numeric(out[col], errors="coerce")

    # Ordenação temporal
    out = out.sort_values([col_id, col_ano]).reset_index(drop=True)

    # Tratamento de duplicidade por município-ano
    duplicadas = out.duplicated([col_id, col_ano]).sum()

    if duplicadas > 0:
        print(f"Atenção: foram encontradas {duplicadas} duplicidades por {col_id}-{col_ano}.")
        print("Consolidando por média nas colunas numéricas e primeiro valor nas demais.")

        cols_nao_valores = [
            col for col in out.columns
            if col not in cols_valores + [col_id, col_ano]
        ]

        agg_dict = {col: "first" for col in cols_nao_valores}

        for col in cols_valores:
            agg_dict[col] = "mean"

        out = (
            out
            .groupby([col_id, col_ano], as_index=False)
            .agg(agg_dict)
            .sort_values([col_id, col_ano])
            .reset_index(drop=True)
        )

    for col in cols_valores:
        prefixo = col

        grupo_col = out.groupby(col_id)[col]

        # Lags
        out[f"{prefixo}_lag_1y"] = grupo_col.shift(1)
        out[f"{prefixo}_lag_2y"] = grupo_col.shift(2)
        # out[f"{prefixo}_lag_3y"] = grupo_col.shift(3)

        # Variações absolutas
        out[f"{prefixo}_var_abs_1y"] = (
            out[col] - out[f"{prefixo}_lag_1y"]
        )

        out[f"{prefixo}_var_abs_2y"] = (
            out[col] - out[f"{prefixo}_lag_2y"]
        )

        # Crescimentos percentuais
        out[f"{prefixo}_crescimento_1y"] = (
            out[col] / out[f"{prefixo}_lag_1y"].replace(0, np.nan) - 1
        )

        out[f"{prefixo}_crescimento_2y_acumulado"] = (
            out[col] / out[f"{prefixo}_lag_2y"].replace(0, np.nan) - 1
        )

        # CAGR
        out[f"{prefixo}_cagr_2y"] = np.where(
            (out[col] > 0) & (out[f"{prefixo}_lag_2y"] > 0),
            (out[col] / out[f"{prefixo}_lag_2y"]) ** (1 / 2) - 1,
            np.nan
        )

        # Log e delta log
        if criar_logs:
            out[f"log_{prefixo}"] = np.where(
                out[col] > 0,
                np.log(out[col]),
                np.nan
            )

            out[f"{prefixo}_delta_log_1y"] = (
                out[f"log_{prefixo}"]
                - out.groupby(col_id)[f"log_{prefixo}"].shift(1)
            )

        # Aceleração do crescimento
        out[f"{prefixo}_crescimento_1y_lag_1y"] = (
            out.groupby(col_id)[f"{prefixo}_crescimento_1y"].shift(1)
        )

        out[f"{prefixo}_aceleracao_crescimento_1y"] = (
            out[f"{prefixo}_crescimento_1y"]
            - out[f"{prefixo}_crescimento_1y_lag_1y"]
        )

        # Ranking dentro do ano
        if criar_ranking_ano:
            out[f"{prefixo}_rank_ano"] = (
                out.groupby(col_ano)[col]
                .rank(method="min", ascending=False)
            )

            out[f"{prefixo}_percentil_ano"] = (
                out.groupby(col_ano)[col]
                .rank(pct=True, ascending=True)
            )

        # Participação no total do ano
        total_ano = out.groupby(col_ano)[col].transform("sum")

        out[f"{prefixo}_share_total_ano"] = (
            out[col] / total_ano.replace(0, np.nan)
        )

    return out


In [41]:
cols_pib_setorial = [
    "vab_agropecuaria",
    "vab_industria",
    "vab_servicos",
    "vab_adm_publica",
    "impostos_liquidos",
]

pib_setorial_var = criar_variaveis_ano_a_ano_multiplas(
    df=pib_setorial_2021,
    col_id="codigo_ibge",
    col_ano="ano",
    cols_valores=cols_pib_setorial,
    criar_logs=True,
    criar_ranking_ano=True,
    criar_rolling=False
)


/Users/lucas.nascimento/Library/Python/3.9/lib/python/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/lucas.nascimento/Library/Python/3.9/lib/python/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/lucas.nascimento/Library/Python/3.9/lib/python/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [42]:
pib_setorial_var.columns


Index(['codigo_ibge', 'ano', 'vab_agropecuaria', 'vab_industria',
       'vab_servicos', 'vab_adm_publica', 'impostos_liquidos',
       'vab_agropecuaria_lag_1y', 'vab_agropecuaria_lag_2y',
       'vab_agropecuaria_var_abs_1y', 'vab_agropecuaria_var_abs_2y',
       'vab_agropecuaria_crescimento_1y',
       'vab_agropecuaria_crescimento_2y_acumulado', 'vab_agropecuaria_cagr_2y',
       'log_vab_agropecuaria', 'vab_agropecuaria_delta_log_1y',
       'vab_agropecuaria_crescimento_1y_lag_1y',
       'vab_agropecuaria_aceleracao_crescimento_1y',
       'vab_agropecuaria_rank_ano', 'vab_agropecuaria_percentil_ano',
       'vab_agropecuaria_share_total_ano', 'vab_industria_lag_1y',
       'vab_industria_lag_2y', 'vab_industria_var_abs_1y',
       'vab_industria_var_abs_2y', 'vab_industria_crescimento_1y',
       'vab_industria_crescimento_2y_acumulado', 'vab_industria_cagr_2y',
       'log_vab_industria', 'vab_industria_delta_log_1y',
       'vab_industria_crescimento_1y_lag_1y',
       'vab_

In [43]:
pib_setorial_var[pib_setorial_var["ano"] == 2021]


,codigo_ibge,ano,vab_agropecuaria,vab_industria,vab_servicos,vab_adm_publica,impostos_liquidos,vab_agropecuaria_lag_1y,vab_agropecuaria_lag_2y,vab_agropecuaria_var_abs_1y,...,impostos_liquidos_crescimento_1y,impostos_liquidos_crescimento_2y_acumulado,impostos_liquidos_cagr_2y,log_impostos_liquidos,impostos_liquidos_delta_log_1y,impostos_liquidos_crescimento_1y_lag_1y,impostos_liquidos_aceleracao_crescimento_1y,impostos_liquidos_rank_ano,impostos_liquidos_percentil_ano,impostos_liquidos_share_total_ano
2,1100015,2021,311468.68,27845.50,1.731216e+05,49596.51,6.848705e+05,203394.42,162668.42,108074.26,...,0.279815,0.468343,0.211752,13.436985,0.246715,0.147310,0.132505,1300.0,0.766786,0.000089
5,1100023,2021,293000.78,407674.51,7.823062e+05,420335.87,2.790958e+06,199722.71,170830.70,93278.07,...,0.106783,0.216158,0.102796,14.841896,0.101457,0.098823,0.007960,408.0,0.926930,0.000362
8,1100031,2021,136659.30,8117.13,4.657925e+04,11192.85,2.272212e+05,81177.10,61501.09,55482.20,...,0.420872,0.695956,0.302289,12.333679,0.351271,0.193602,0.227269,2862.0,0.486355,0.000029
11,1100049,2021,341314.54,236801.00,6.291094e+05,384627.38,2.407879e+06,236214.95,193076.51,105099.59,...,0.072837,0.199177,0.095069,14.694257,0.070307,0.117762,-0.044924,464.0,0.916876,0.000312
14,1100056,2021,151690.48,27433.02,1.233646e+05,131905.54,6.111561e+05,94757.64,70991.74,56932.84,...,0.196685,0.421857,0.192417,13.323108,0.179555,0.188164,0.008521,1409.0,0.747217,0.000079
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16697,5222005,2021,415874.61,43894.84,7.576018e+04,66056.83,7.919593e+05,237993.62,174224.62,177880.99,...,0.533002,0.754733,0.324663,13.582265,0.427228,0.144638,0.388364,1169.0,0.790305,0.000103
16700,5222054,2021,194666.87,87665.58,5.265496e+04,31900.27,4.483098e+05,158818.12,100351.51,35848.75,...,0.161220,0.492924,0.221853,13.013240,0.149471,0.285651,-0.124432,1789.0,0.678995,0.000058
16703,5222203,2021,58414.42,30276.01,3.673052e+04,14192.65,1.657312e+05,28357.52,24110.66,30056.90,...,0.409065,0.599005,0.264518,12.018123,0.342926,0.134799,0.274266,3425.0,0.385278,0.000021
16706,5222302,2021,157949.39,57342.31,3.314975e+04,12865.04,2.951700e+05,115113.53,104542.03,42835.86,...,0.409767,0.476042,0.214925,12.595307,0.343424,0.047011,0.362756,2382.0,0.572531,0.000038


In [44]:
pib_setorial_var[(pib_setorial_var["codigo_ibge"] == "3509502")&(pib_setorial_var["ano"] == 2021)]


,codigo_ibge,ano,vab_agropecuaria,vab_industria,vab_servicos,vab_adm_publica,impostos_liquidos,vab_agropecuaria_lag_1y,vab_agropecuaria_lag_2y,vab_agropecuaria_var_abs_1y,...,impostos_liquidos_crescimento_1y,impostos_liquidos_crescimento_2y_acumulado,impostos_liquidos_cagr_2y,log_impostos_liquidos,impostos_liquidos_delta_log_1y,impostos_liquidos_crescimento_1y_lag_1y,impostos_liquidos_aceleracao_crescimento_1y,impostos_liquidos_rank_ano,impostos_liquidos_percentil_ano,impostos_liquidos_share_total_ano
10127,3509502,2021,170416.99,11136729.65,5182393.18,13827274.6,59119635.57,165109.29,119398.59,5307.7,...,0.112877,0.090734,0.044382,17.895074,0.106949,-0.019898,0.132775,13.0,0.997846,0.007664


In [45]:
pib_setorial_var_2021 = pib_setorial_var[pib_setorial_var["ano"] == 2021]
pib_setorial_var_2021.head()


,codigo_ibge,ano,vab_agropecuaria,vab_industria,vab_servicos,vab_adm_publica,impostos_liquidos,vab_agropecuaria_lag_1y,vab_agropecuaria_lag_2y,vab_agropecuaria_var_abs_1y,...,impostos_liquidos_crescimento_1y,impostos_liquidos_crescimento_2y_acumulado,impostos_liquidos_cagr_2y,log_impostos_liquidos,impostos_liquidos_delta_log_1y,impostos_liquidos_crescimento_1y_lag_1y,impostos_liquidos_aceleracao_crescimento_1y,impostos_liquidos_rank_ano,impostos_liquidos_percentil_ano,impostos_liquidos_share_total_ano
2,1100015,2021,311468.68,27845.50,173121.63,49596.51,684870.51,203394.42,162668.42,108074.26,...,0.279815,0.468343,0.211752,13.436985,0.246715,0.147310,0.132505,1300.0,0.766786,0.000089
5,1100023,2021,293000.78,407674.51,782306.22,420335.87,2790958.40,199722.71,170830.70,93278.07,...,0.106783,0.216158,0.102796,14.841896,0.101457,0.098823,0.007960,408.0,0.926930,0.000362
8,1100031,2021,136659.30,8117.13,46579.25,11192.85,227221.25,81177.10,61501.09,55482.20,...,0.420872,0.695956,0.302289,12.333679,0.351271,0.193602,0.227269,2862.0,0.486355,0.000029
11,1100049,2021,341314.54,236801.00,629109.37,384627.38,2407878.53,236214.95,193076.51,105099.59,...,0.072837,0.199177,0.095069,14.694257,0.070307,0.117762,-0.044924,464.0,0.916876,0.000312
14,1100056,2021,151690.48,27433.02,123364.62,131905.54,611156.13,94757.64,70991.74,56932.84,...,0.196685,0.421857,0.192417,13.323108,0.179555,0.188164,0.008521,1409.0,0.747217,0.000079


In [46]:
pib_setorial_var_2021.to_parquet(TRUSTED_DIR / "sidra_pib_setorial_anoxano.parquet", index=False)


## Montar ABT municipal IBGE

In [47]:
def montar_abt_ibge_painel(
    trusted_dir: Path,
    analytics_dir: Path,
    arquivo_dim_municipio: str = "dim_municipio.parquet",
    arquivo_pib_total: str = "sidra_pib_anoxano.parquet",
    arquivo_pib_setorial: str = "sidra_pib_setorial_anoxano.parquet",
    arquivo_pop_censo: Optional[str] = "populacao_censos_2010_2022_municipio.parquet",
    salvar: bool = True
) -> pd.DataFrame:
    """
    Monta ABT municipal em formato painel cidade-ano usando:
    - dimensão de municípios
    - população estimada dos últimos 5 anos
    - PIB municipal total dos últimos 5 anos
    - PIB setorial dos últimos 5 anos
    - população do Censo 2022, opcional, repetida por município

    A granularidade final é:
        codigo_ibge × ano
    """

    # -----------------------------
    # 1. Leitura das bases
    # -----------------------------
    dim = pd.read_parquet(trusted_dir / arquivo_dim_municipio)
    pop_censo = pd.read_parquet(trusted_dir / arquivo_pop_censo)
    pib_total = pd.read_parquet(trusted_dir / arquivo_pib_total).drop(columns=["ano"])
    pib_setorial = pd.read_parquet(trusted_dir / arquivo_pib_setorial).drop(columns=["ano"])

    # -----------------------------
    # 2. Padronização
    # -----------------------------
    dim = dim.copy()
    dim["codigo_ibge"] = dim["codigo_ibge"].astype(str).str.zfill(7)

    # -----------------------------
    # 3. Construção do painel base
    # -----------------------------
    # A população estimada define o esqueleto cidade-ano.
    # Em tese, deve trazer todos os municípios em todos os anos disponíveis.
    
    painel = pop_censo.copy()

    # -----------------------------
    # 4. Junção com PIB total e PIB setorial
    # -----------------------------
    painel = (
        painel
        .merge(
            pib_total,
            on=["codigo_ibge"],
            how="left"
        )
        .merge(
            pib_setorial,
            on=["codigo_ibge"],
            how="left"
        )
    )

    # -----------------------------
    # 5. Junção com dimensão municipal
    # -----------------------------
    abt = (
        painel
        .merge(
            dim,
            on="codigo_ibge",
            how="left"
        )
    )

    # -----------------------------
    # 7. Organização das colunas
    # -----------------------------
    colunas_identificacao = [
        "codigo_ibge",
        "ano",
        "municipio",
        "uf",
        "nome_uf",
        "regiao",
        "sigla_regiao",
        "regiao_imediata",
        "regiao_intermediaria"
    ]

    colunas_identificacao = [
        col for col in colunas_identificacao if col in abt.columns
    ]

    demais_colunas = [
        col for col in abt.columns if col not in colunas_identificacao
    ]

    abt = abt[colunas_identificacao + demais_colunas]

    # -----------------------------
    # 8. Variáveis derivadas
    # -----------------------------

    # Logs para reduzir assimetria
    if "populacao_censo_2022" in abt.columns:
        abt["log_populacao_censo_2022"] = np.log1p(abt["populacao_censo_2022"])

    if "pib_corrente" in abt.columns:
        abt["log_pib_corrente"] = np.log1p(abt["pib_corrente"])

    # PIB per capita aproximado
    # Atenção: dependendo do retorno SIDRA, pib_corrente pode estar em mil reais.
    # Se estiver em mil reais, usamos * 1000.
    if {"pib_corrente", "populacao_censo_2022"}.issubset(abt.columns):
        abt["pib_per_capita_aprox"] = np.where(
            abt["populacao_censo_2022"] > 0,
            (abt["pib_corrente"] * 1000) / abt["populacao_censo_2022"],
            np.nan
        )

    # Participações setoriais no PIB/VAB
    cols_setoriais = [
        "vab_agropecuaria",
        "vab_industria",
        "vab_servicos",
        "vab_adm_publica"
    ]

    cols_setoriais_existentes = [
        col for col in cols_setoriais if col in abt.columns
    ]

    if len(cols_setoriais_existentes) > 0:
        abt["vab_total_setorial"] = abt[cols_setoriais_existentes].sum(axis=1, min_count=1)

        for col in cols_setoriais_existentes:
            abt[f"share_{col}"] = np.where(
                abt["vab_total_setorial"] > 0,
                abt[col] / abt["vab_total_setorial"],
                np.nan
            )

    # Razão serviços + administração pública
    if {"vab_servicos", "vab_adm_publica"}.issubset(abt.columns):
        abt["vab_servicos_total"] = abt["vab_servicos"] + abt["vab_adm_publica"]

        if "vab_total_setorial" in abt.columns:
            abt["share_servicos_total"] = np.where(
                abt["vab_total_setorial"] > 0,
                abt["vab_servicos_total"] / abt["vab_total_setorial"],
                np.nan
            )

    # Crescimento populacional e econômico ano contra ano
    abt = abt.sort_values(["codigo_ibge", "ano"]).reset_index(drop=True)

    # -----------------------------
    # 9. Salvar
    # -----------------------------
    if salvar:
        analytics_dir.mkdir(parents=True, exist_ok=True)

        abt.to_parquet(
            analytics_dir / "abt_municipios_ibge_painel.parquet",
            index=False
        )

        # abt.to_csv(
        #     analytics_dir / "abt_municipios_ibge_painel.csv",
        #     index=False,
        #     encoding="utf-8"
        # )

    return abt


In [48]:
abt_ibge = montar_abt_ibge_painel(
    trusted_dir=TRUSTED_DIR,
    analytics_dir=ANALYTICS_DIR
)

abt_ibge.head()


,codigo_ibge,ano,municipio,uf,nome_uf,regiao,sigla_regiao,regiao_imediata,regiao_intermediaria,populacao_censo_2022,...,codigo_regiao_intermediaria,log_populacao_censo_2022,pib_per_capita_aprox,vab_total_setorial,share_vab_agropecuaria,share_vab_industria,share_vab_servicos,share_vab_adm_publica,vab_servicos_total,share_servicos_total
0,1100015,2022,Alta Floresta D'Oeste,RO,Rondônia,Norte,N,Cacoal,Ji-Paraná,21494,...,1102,9.975576,48680.680655,562032.32,0.554183,0.049544,0.308028,0.088245,222718.14,0.396273
1,1100023,2022,Ariquemes,RO,Rondônia,Norte,N,Ariquemes,Porto Velho,96833,...,1101,11.480753,45271.914946,1903317.38,0.153942,0.214192,0.411022,0.220844,1202642.09,0.631866
2,1100031,2022,Cabixi,RO,Rondônia,Norte,N,Vilhena,Ji-Paraná,5351,...,1102,8.585226,56219.792562,202548.53,0.674699,0.040075,0.229966,0.055260,57772.10,0.285226
3,1100049,2022,Cacoal,RO,Rondônia,Norte,N,Cacoal,Ji-Paraná,86887,...,1102,11.372375,44292.792938,1591852.29,0.214413,0.148758,0.395206,0.241623,1013736.75,0.636828
4,1100056,2022,Cerejeiras,RO,Rondônia,Norte,N,Vilhena,Ji-Paraná,15890,...,1102,9.673508,63334.766520,434393.66,0.349200,0.063152,0.283993,0.303654,255270.16,0.587647


In [49]:
abt_ibge.columns.to_list()


['codigo_ibge',
 'ano',
 'municipio',
 'uf',
 'nome_uf',
 'regiao',
 'sigla_regiao',
 'regiao_imediata',
 'regiao_intermediaria',
 'populacao_censo_2022',
 'populacao_censo_2010',
 'crescimento_populacao_censo_2010_2022',
 'cagr_populacao_censo_2010_2022',
 'pib_corrente',
 'municipio_sidra',
 'pib_corrente_lag_1y',
 'pib_corrente_lag_2y',
 'pib_corrente_lag_3y',
 'pib_corrente_var_abs_1y',
 'pib_corrente_var_abs_2y',
 'pib_corrente_var_abs_3y',
 'pib_corrente_crescimento_1y',
 'pib_corrente_crescimento_2y_acumulado',
 'pib_corrente_crescimento_3y_acumulado',
 'pib_corrente_cagr_2y',
 'pib_corrente_cagr_3y',
 'log_pib_corrente',
 'pib_corrente_delta_log_1y',
 'pib_corrente_crescimento_1y_lag_1y',
 'pib_corrente_aceleracao_crescimento_1y',
 'pib_corrente_media_movel_3y',
 'pib_corrente_desvio_movel_3y',
 'pib_corrente_coef_var_movel_3y',
 'pib_corrente_indice_base_primeiro_ano',
 'pib_corrente_crescimento_acumulado_primeiro_ano',
 'pib_corrente_rank_ano',
 'pib_corrente_percentil_ano',


In [50]:
abt_ibge[abt_ibge["municipio"] == "Campinas"]


,codigo_ibge,ano,municipio,uf,nome_uf,regiao,sigla_regiao,regiao_imediata,regiao_intermediaria,populacao_censo_2022,...,codigo_regiao_intermediaria,log_populacao_censo_2022,pib_per_capita_aprox,vab_total_setorial,share_vab_agropecuaria,share_vab_industria,share_vab_servicos,share_vab_adm_publica,vab_servicos_total,share_servicos_total
3375,3509502,2022,Campinas,SP,São Paulo,Sudeste,SE,Campinas,Campinas,1139047,...,3510,13.945703,80741.469246,30316814.42,0.005621,0.367345,0.170941,0.456093,19009667.78,0.627034


In [51]:
abt_ibge[abt_ibge["municipio"] == "Paranaguá"]


,codigo_ibge,ano,municipio,uf,nome_uf,regiao,sigla_regiao,regiao_imediata,regiao_intermediaria,populacao_censo_2022,...,codigo_regiao_intermediaria,log_populacao_censo_2022,pib_per_capita_aprox,vab_total_setorial,share_vab_agropecuaria,share_vab_industria,share_vab_servicos,share_vab_adm_publica,vab_servicos_total,share_servicos_total
4164,4118204,2022,Paranaguá,PR,Paraná,Sul,S,Paranaguá,Curitiba,145829,...,4101,11.890197,104860.505592,6575536.73,0.002535,0.469119,0.139883,0.388463,3474157.95,0.528346


In [52]:
abt_ibge.to_parquet(ANALYTICS_DIR / "abt_municipios_ibge_basica.parquet", index=False)


In [53]:
abt_ibge.columns.to_list()


['codigo_ibge',
 'ano',
 'municipio',
 'uf',
 'nome_uf',
 'regiao',
 'sigla_regiao',
 'regiao_imediata',
 'regiao_intermediaria',
 'populacao_censo_2022',
 'populacao_censo_2010',
 'crescimento_populacao_censo_2010_2022',
 'cagr_populacao_censo_2010_2022',
 'pib_corrente',
 'municipio_sidra',
 'pib_corrente_lag_1y',
 'pib_corrente_lag_2y',
 'pib_corrente_lag_3y',
 'pib_corrente_var_abs_1y',
 'pib_corrente_var_abs_2y',
 'pib_corrente_var_abs_3y',
 'pib_corrente_crescimento_1y',
 'pib_corrente_crescimento_2y_acumulado',
 'pib_corrente_crescimento_3y_acumulado',
 'pib_corrente_cagr_2y',
 'pib_corrente_cagr_3y',
 'log_pib_corrente',
 'pib_corrente_delta_log_1y',
 'pib_corrente_crescimento_1y_lag_1y',
 'pib_corrente_aceleracao_crescimento_1y',
 'pib_corrente_media_movel_3y',
 'pib_corrente_desvio_movel_3y',
 'pib_corrente_coef_var_movel_3y',
 'pib_corrente_indice_base_primeiro_ano',
 'pib_corrente_crescimento_acumulado_primeiro_ano',
 'pib_corrente_rank_ano',
 'pib_corrente_percentil_ano',


## Leitura de Dados

In [54]:
df_ibge_painel = pd.read_parquet(ANALYTICS_DIR / "abt_municipios_ibge_painel.parquet")[
    ["codigo_ibge", 
    "ano", "municipio",
    "populacao_estimada",
    "pib_corrente",
    "vab_agropecuaria",
    "vab_industria",
    "vab_servicos",
    "vab_adm_publica",
    "impostos_liquidos",
    "populacao_censo_2022",
    "populacao_censo_2022",
    "log_populacao_estimada",
    "log_pib_corrente",
    "pib_per_capita_aprox",
    "vab_total_setorial",
    "share_vab_agropecuaria",
    "share_vab_industria",
    "share_vab_servicos",
    "share_vab_adm_publica",
    "vab_servicos_total",
    "share_servicos_total",
    "crescimento_populacao_1y",
    "crescimento_pib_corrente_1y",
    "crescimento_pib_per_capita_1y",
    "crescimento_vab_agropecuaria_1y",
    "crescimento_vab_industria_1y",
    "crescimento_vab_servicos_1y",
    "crescimento_vab_adm_publica_1y",
    ]
]
df_ibge_painel


KeyError: "['populacao_estimada', 'log_populacao_estimada', 'crescimento_populacao_1y', 'crescimento_pib_corrente_1y', 'crescimento_pib_per_capita_1y', 'crescimento_vab_agropecuaria_1y', 'crescimento_vab_industria_1y', 'crescimento_vab_servicos_1y', 'crescimento_vab_adm_publica_1y'] not in index"

In [ ]:
df_ibge_painel[df_ibge_painel["codigo_ibge"] == "4118204"][
    ['codigo_ibge', 'ano', 'municipio', 'uf', 'nome_uf', 'regiao',
       'sigla_regiao', 'regiao_imediata', 'regiao_intermediaria','municipio_norm',
       'codigo_uf', 'codigo_regiao', 'codigo_regiao_imediata', 'codigo_regiao_intermediaria']
       ]


In [ ]:
df_ibge_painel[df_ibge_painel["codigo_ibge"] == "4118204"][
    ['codigo_ibge', 'ano', 'populacao_censo_2022', 'log_populacao_estimada', 'log_pib_corrente',
       'pib_per_capita_aprox', 'vab_total_setorial', 'share_vab_agropecuaria',
       'share_vab_industria', 'share_vab_servicos', 'share_vab_adm_publica',
       'vab_servicos_total', 'share_servicos_total',
       'crescimento_populacao_1y', 'crescimento_pib_corrente_1y',]
       ]


In [ ]:
df_ibge_painel[df_ibge_painel["codigo_ibge"] == "4118204"]


In [ ]:
df_ibge_painel.columns
